# SONICS 음악 탐지 모델 검증
T4 GPU에서 모두 실행하고 sonics_check_bundle.zip 하나를 선택하세요.
새 패키지 timm 설치는 사용자 승인에 따라 Colab에만 수행합니다. 학습하거나 DACON에 제출하지 않습니다.
5초 모델이 주 비교 대상이며 120초 모델은 길이 불일치를 확인하는 보조 실험입니다. 결과: sonics_results.zip


In [ ]:
from google.colab import files
from pathlib import Path
import torch, sys, json, hashlib, zipfile, io, tempfile, shutil, subprocess, os
assert torch.cuda.is_available(), "런타임 유형을 T4 GPU로 바꾸세요."
print(torch.cuda.get_device_name(0), torch.__version__, sys.version)
uploaded=files.upload()
assert len(uploaded)==1, "sonics_check_bundle.zip 하나만 선택하세요."
blob=next(iter(uploaded.values()))
assert hashlib.sha256(blob).hexdigest()=="60fa327ecaf8e78db456c6cc07a9f3d2c40eefc2ef8ba6e14d0ef0f5354361f0", "ZIP 버전이 다릅니다."
WORK=Path(tempfile.mkdtemp(prefix="sonics_check_",dir="/content"))
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for name in z.namelist():
        assert (WORK/name).resolve().is_relative_to(WORK.resolve())
    z.extractall(WORK)
plan=json.loads((WORK/"plan.json").read_text())
for name,sha in plan["file_sha256"].items():
    assert hashlib.sha256((WORK/name).read_bytes()).hexdigest()==sha, name
RESULTS=WORK/"sonics_results"
RESULTS.mkdir()
print("128개 입력과 실행 코드 무결성 확인 완료")


In [ ]:
# torch/torchvision을 임의로 업그레이드하지 않는다.
installed=subprocess.run([sys.executable,"-m","pip","install","--no-deps","timm==1.0.15"],capture_output=True,text=True)
(RESULTS/"install.log").write_text(installed.stdout+installed.stderr)
print((installed.stdout+installed.stderr)[-4000:])
installed.check_returncode()
check=subprocess.run([sys.executable,"-c","import timm, torch, torchvision, torchaudio, librosa, soundfile, huggingface_hub; from sonics import HFAudioClassifier; print('의존성 검사 통과', timm.__version__)"],cwd=WORK,capture_output=True,text=True)
(RESULTS/"preflight.log").write_text(check.stdout+check.stderr)
print(check.stdout+check.stderr)
check.check_returncode()
assert shutil.which("ffmpeg"), "ffmpeg가 없습니다."
(RESULTS/"environment.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)+"\n"+sys.version)


In [ ]:
from huggingface_hub import hf_hub_download
for variant,entry in plan["models"].items():
    cached=Path(hf_hub_download(repo_id=entry["repo"],filename="pytorch_model.bin",revision=entry["revision"]))
    assert cached.stat().st_size==entry["weights_size"]
    assert hashlib.sha256(cached.read_bytes()).hexdigest()==entry["weights_sha256"], variant
    shutil.copy2(cached,WORK/"weights"/variant/"pytorch_model.bin")
    print(variant,"가중치 해시 검사 통과")


In [ ]:
# 추론은 로컬 가중치를 사용하며 HF 네트워크 다운로드를 금지한다.
env=os.environ.copy()
env["HF_HUB_OFFLINE"]="1"
env["TRANSFORMERS_OFFLINE"]="1"
try:
    with (RESULTS/"run.log").open("w",encoding="utf-8") as log:
        process=subprocess.Popen([sys.executable,"-u","run_sonics_check.py","--work",str(WORK)],cwd=WORK,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in process.stdout:
                print(line,end="")
                log.write(line)
                log.flush()
            code=process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait()
    assert code==0, "검증이 중단됐습니다. 결과 ZIP과 오류 내용을 보내주세요."
finally:
    archive=shutil.make_archive("/content/sonics_results","zip",RESULTS)
    files.download(archive)
